In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
from ast import literal_eval

df = pd.read_csv('data/matches_processed.csv')
df["Player"] = df["Player"].apply(literal_eval)

In [3]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

class WinPredictionDataset(Dataset):
    def __init__(self, players, result):
        self.players = players
        self.results = torch.tensor(result, dtype=torch.float32)
        
        # Flatten all tokens to build vocabulary
        all_tokens = [item for row in players for subsublist in row for item in subsublist]
        self.tokenizer = LabelEncoder()
        self.tokenizer.fit(all_tokens)  # Fit on all possible tokens
        
        # Pre-encode all data during init (more efficient)
        self.encoded_players = [
            [
                self.tokenizer.transform(subsublist) 
                for subsublist in row
            ] 
            for row in players
        ]
        
    def __len__(self):
        return len(self.players)

    def __getitem__(self, idx):
        return {
            "players": torch.tensor(np.array(self.encoded_players[idx]), dtype=torch.long),  # Shape: [10, 5]
            "results": self.results[idx]  # Shape: [1]
        }

In [7]:
import torch
import torch.nn as nn
import math

class SinusoidalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 50):  # 10×5=50
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)  # [0, 1, ..., 49]
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dims: sine
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dims: cosine
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)] 

class WinPredictionModel(nn.Module):
    def __init__(self, vocab_size=511, embedding_dim=64, dropout=0.5):
        super().__init__()
        # (1) Stat embeddings (from token IDs)
        self.stat_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # (2) LEARNABLE role embeddings (5 roles: Top, Jungle, Mid, etc.)
        self.role_embedding = nn.Embedding(5, embedding_dim)  # 5 roles
        
        # (3) Team embeddings (allied=0, enemy=1)
        self.team_embedding = nn.Embedding(2, embedding_dim)
        
        # (4) CNN layers
        self.conv = nn.Sequential(
            nn.Conv2d(embedding_dim, 128, kernel_size=(5, 5), padding=(2, 2)), # [batch, 128, 10, 5]
            nn.BatchNorm2d(128),  # Stabilize training
            nn.Dropout(dropout),
            nn.ReLU(),
            
            nn.Conv2d(128, 128, kernel_size=(5, 5), padding=(2, 2)),  # [batch, 128, 10, 5]
            nn.BatchNorm2d(128),  # Stabilize training
            nn.Dropout(dropout),
            nn.ReLU()
        )
        
        # (5) New Positional Encoding + Transformer encoder
        self.pos_encoder = SinusoidalEncoding(d_model=128)
        
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=128, nhead=8, dim_feedforward=512, dropout=dropout, activation='gelu', batch_first=True
            ),
            num_layers=3,
            enable_nested_tensor=True 
        )
        
        # (6) Final classifier
        # Classifier
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: [batch, 10 players, 5 stats]
        batch_size = x.size(0)
        
        # (1) Embed stats -> [batch, 10, 5, 64]
        embedded = self.stat_embedding(x)
        
        # (2) Add LEARNABLE role embeddings
        role_ids = torch.arange(5).repeat(2).to(x.device)  # [0,1,2,3,4, 0,1,2,3,4] (10 players)
        role_embedded = self.role_embedding(role_ids)      # [10, 64]
        embedded += role_embedded.unsqueeze(0).unsqueeze(2)  # [batch, 10, 5, 64]
        
        # (3) Add team embeddings
        team_ids = torch.cat([
            torch.zeros(batch_size, 5, dtype=torch.long),  # allied
            torch.ones(batch_size, 5, dtype=torch.long)    # enemy
        ], dim=1).to(x.device)
        team_embedded = self.team_embedding(team_ids).unsqueeze(2)  # [batch, 10, 1, 64]
        embedded += team_embedded
        
        # Reshape for CNN: [batch, 64, 10, 5]
        embedded = embedded.permute(0, 3, 1, 2)
        
        # (4) Apply CNN layers
        x = self.conv(embedded) # [batch, 128, 10, 5]
        
        # Update x shape since transformer encoder expects a sequence
        x = x.permute(0, 2, 3, 1)  # [batch, 10, 5, 128]
        x = x.reshape(x.shape[0], x.shape[1] * x.shape[2], x.shape[3])  # [batch, 50, 128]
        
        # (5) Add one more time positional encodings and pass through transformer encoder
        x = self.pos_encoder(x)
        x = self.encoder(x)
        
        # (6) Global average pooling and pass through classifier
        x = x.mean(dim=1)
        return self.head(x)
        
        

In [5]:
from sklearn.metrics import confusion_matrix, accuracy_score

def eval_model(model, val_loader, criterion, device):
    model.eval()
    
    all_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            players = batch["players"].to(device)
            labels = batch["results"].to(device)
            
            outputs = model(players)
            loss = criterion(outputs, labels.unsqueeze(1)) 
            all_loss += loss.item()
            
            preds = np.round(outputs.cpu().numpy())
            
            all_preds.extend(preds.flatten().tolist())
            all_labels.extend(labels.cpu().numpy().flatten().tolist())
            
    average_loss = all_loss / len(val_loader)            
    accuracy = accuracy_score(all_labels, all_preds)
    conf_matrix = confusion_matrix(all_labels, all_preds)  # Call the function from sklearn.metrics
    return average_loss, accuracy, conf_matrix

In [10]:
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch
import numpy as np
import visualize

torch.manual_seed(42)
np.random.seed(42)

n_folds = 5
num_epochs = 50
learning_rate = 2e-4
batch_size = 128

dataset = WinPredictionDataset(df["Player"].values, df["Win"].to_numpy())

kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

fold_train_losses = []
fold_val_losses = []
fold_accuracies = []
fold_conf_matrices = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"Fold {fold + 1}/{kf.n_splits}")
    
    trainset = torch.utils.data.Subset(dataset, train_idx)
    valset = torch.utils.data.Subset(dataset, val_idx)

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)
    
    # Initialize model, optimizer, and loss function
    model = WinPredictionModel(vocab_size=dataset.tokenizer.classes_.size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()

    train_losses = []
    val_losses = []
    accuracies = []
    conf_matrices = []

    for epoch in range(num_epochs):
        train_loss = 0
        model.train()
        for batch in train_loader:
            players = batch["players"].to(device)
            results = batch["results"].to(device)
            
            optimizer.zero_grad()
            outputs = model(players)
            loss = criterion(outputs, results.unsqueeze(1))  # Ensure results are the same shape as outputs
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        average_train_loss = train_loss / len(train_loader)
        average_val_loss, accuracy, conf_matrix = eval_model(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_train_loss}/{average_val_loss}, Validation accuracy: {accuracy}")
        
        train_losses.append(average_train_loss)
        val_losses.append(average_val_loss)
        accuracies.append(accuracy)
        
    fold_train_losses.append(train_losses)
    fold_val_losses.append(val_losses)
    fold_accuracies.append(accuracies)
    fold_conf_matrices.append(conf_matrix)
    
# Average the results across folds
train_losses = np.mean(fold_train_losses, axis=0)
val_losses = np.mean(fold_val_losses, axis=0)
accuracies = np.mean(fold_accuracies, axis=0)
conf_matrices = np.mean(fold_conf_matrices, axis=0)

visualize.loss(train_losses, val_loss=val_losses, title=f"Train and Validation Losses across {n_folds} folds")
visualize.accuracy(accuracies, title=f"Validation Accuracy across {n_folds} folds")
visualize.confusion_matrix(conf_matrices, title=f"Confusion Matrix across {n_folds} folds")

Using device: cuda
Fold 1/5
Epoch 1/50, Loss: 0.6934734459907289/0.6927318200469017, Validation accuracy: 0.512
Epoch 2/50, Loss: 0.6928157948312306/0.6927519515156746, Validation accuracy: 0.512
Epoch 3/50, Loss: 0.6924616240319752/0.6932028569281101, Validation accuracy: 0.512
Epoch 4/50, Loss: 0.6919463976981148/0.6937100477516651, Validation accuracy: 0.512
Epoch 5/50, Loss: 0.6919756910157582/0.6933919303119183, Validation accuracy: 0.512
Epoch 6/50, Loss: 0.6916016557859996/0.6927138566970825, Validation accuracy: 0.512
Epoch 7/50, Loss: 0.691336402817378/0.6925755478441715, Validation accuracy: 0.512
Epoch 8/50, Loss: 0.6908833223675924/0.6929027289152145, Validation accuracy: 0.505
Epoch 9/50, Loss: 0.6898261554657467/0.6919361427426338, Validation accuracy: 0.511
Epoch 10/50, Loss: 0.6869258681933085/0.6975931487977505, Validation accuracy: 0.5195
Epoch 11/50, Loss: 0.6867078694086226/0.6933908872306347, Validation accuracy: 0.511
Epoch 12/50, Loss: 0.6846741239229838/0.689686